In [ ]:
import pandas as pd
import numpy as np
import pickle
import os

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

pd.options.display.float_format = '{:,.2f}'.format

In [ ]:
sns.set_style('darkgrid')

In [ ]:
embedded_features = pd.read_csv('../data/embedded_features.csv', index_col=0)
songs = pd.read_csv('../data/songs_data_edited.csv', index_col=0)
# returns 328D embedded feature vectors
embedded_features.index.name = 'track_id'

embedded_features = songs[['track_id','artist_id']].merge(embedded_features, how='outer',on='track_id',indicator=True)
embedded_features = embedded_features.groupby('artist_id').mean(numeric_only=True)
embedded_features.columns = [f'Feature {i}' for i in embedded_features.columns]

In [ ]:
features = embedded_features.columns
X = embedded_features.values

In [ ]:
# scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# PCA Elbow Method
components_range = range(1, 51)
variances = []
for i in components_range:
    pca = PCA(n_components=i)
    pca.fit(X_scaled)
    variance = sum(pca.explained_variance_ratio_)
    variances.append(variance)

plt.figure(figsize=(14,10))
sns.lineplot(x=components_range, y=variances, color='r', marker='o')
plt.title('Explained Variance Ratio by Number of Components')
plt.show()

In [ ]:
# PCA
n_components = 20
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_scaled)

In [ ]:
# KMeans Elbow Method
cluster_range = range(2, 11)
silhouette_scores = []
for i in cluster_range:
    kmeans = KMeans(n_clusters=i, random_state=1)
    labels = kmeans.fit_predict(X_pca)
    labels = np.array([i+1 for i in labels])
    s_score = silhouette_score(X_pca, labels)
    silhouette_scores.append(s_score)

plt.figure(figsize=(14,10))
sns.lineplot(x=cluster_range, y=silhouette_scores, color='r', marker='o')
plt.title('Silhouette Score by Number of KMeans')
plt.show()

In [ ]:
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=1)
labels = kmeans.fit_predict(X_pca)
labels = np.array([i+1 for i in labels])
inertia = kmeans.inertia_
s_score = silhouette_score(X_pca, labels)

In [ ]:
MODELS_HOME = '../models/'
file_name = 'kmeans_model.pkl'
file_path = MODELS_HOME + file_name

if not os.path.exists(file_path):
    with open(file_path, 'wb') as file:
        pickle.dump(kmeans, file)
    print("Model saved.")
else:
    print("Model already exists.")

In [ ]:
# Cluster Analysis visualization with 2 PCA components
n_components = 2
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_scaled)
plot_df = pd.DataFrame(index=embedded_features.index, data=X_pca, columns=['PC1','PC2'])
plot_df = plot_df.merge(songs[['artist_id','name']].drop_duplicates(), on='artist_id', how='outer')
plot_df['Cluster Group'] = [str(x) for x in labels]
plot_df = plot_df.sort_values('Cluster Group')

# simplified projection of seeing data using 2 PCA components (explains ~50% of variance)
fig = px.scatter(plot_df, x='PC1',y='PC2',
                 color='Cluster Group',
                 hover_data=['name'],
                 title='K-means Cluster Visualization (PCA 2D Projection)')
fig.update_traces(marker=dict(size=8))
fig.update_layout(width=1000, height=800)
fig.show()

## EDA

In [ ]:
classifier_features = pd.read_csv('../data/classifier_features.csv')
classifier = songs[['artist_id','name','track_id','popularity']].merge(classifier_features, on='track_id', how='outer')
classifier = classifier.groupby(['artist_id','name']).mean(numeric_only=True)
classifier = plot_df.merge(classifier, left_on=['artist_id','name'], right_index=True, how='outer')
classifier = classifier.sort_values('Cluster Group')

print(classifier.corr(numeric_only=True)['instrumental'])

In [ ]:
fig = px.scatter(classifier,
                 x='raphiphop',
                 y='pop',
                 color='Cluster Group',
                 hover_data=['name'],
                 title='Rap vs. Pop Probabilities By Artist')
fig.update_traces(marker=dict(size=8))
fig.update_layout(width=1000, height=800)
fig.show()

In [ ]:
correlation = classifier.corr(numeric_only=True)
correlation = correlation[['PC1','PC2']]

plt.figure(figsize=(14,10))
sns.heatmap(correlation, cmap='coolwarm', annot=True)
plt.title('Correlation By Classification Feature For PC1 and PC2')
plt.show()

In [ ]:
plt.figure(figsize=(14,10))
sns.boxplot(classifier, x='Cluster Group', y='acoustic')
plt.title('Acousticness by Cluster Group')
plt.show()